## 1. Comprendre BERT et XLM-RoBERTa

**BERT (Bidirectional Encoder Representations from Transformers)** est un modèle de langage pré-entraîné développé par Google. Il est 'bidirectionnel' car il prend en compte le contexte des mots dans les deux directions (gauche et droite) pour comprendre leur signification, contrairement aux modèles précédents qui ne lisaient que dans un sens. BERT est entraîné sur une grande quantité de texte non annoté pour deux tâches principales :
- **Masquage de Langue (Masked Language Model - MLM)** : Il prédit des mots masqués dans une phrase.
- **Prédiction de la Prochaine Phrase (Next Sentence Prediction - NSP)** : Il détermine si deux phrases sont consécutives.

**XLM-RoBERTa (Cross-lingual Language Model - Robustly Optimized BERT Pretraining Approach)** est une extension multilingue de RoBERTa (une version optimisée de BERT). XLM-RoBERTa est entraîné sur des données textuelles provenant de centaines de langues simultanément. Cela lui permet de comprendre et de générer du texte dans plusieurs langues et de transférer des connaissances entre elles, ce qui est particulièrement utile pour les tâches NLP multilingues où les données étiquetées peuvent être rares dans certaines langues.

Ces modèles utilisent un mécanisme appelé 'attention' pour pondérer l'importance des différents mots dans une phrase lorsqu'ils traitent le texte. La **tokenisation** est la première étape, où le texte est divisé en unités plus petites (tokens) que le modèle peut comprendre.

## 2. Tokenisation de texte

Nous allons maintenant utiliser les tokenizers de BERT et XLM-RoBERTa pour convertir des phrases en entrées tokenisées. Cela inclut la compréhension des `input_ids`, `attention_mask` et `token_type_ids`.

In [1]:
from transformers import BertTokenizer, XLMRobertaTokenizer

# Initialisation des tokenizers
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

# Exemple de phrase
sentence = "Ceci est un exemple de phrase pour la tokenisation."
sentence_en = "This is an example sentence for tokenization."

# Tokenisation avec BertTokenizer
print("\n--- BertTokenizer (bert-base-uncased) ---")
bert_tokens = bert_tokenizer.tokenize(sentence_en)
print(f"Tokens (BERT): {bert_tokens}")

bert_encoded_plus = bert_tokenizer.encode_plus(
    sentence_en,
    add_special_tokens=True, # Ajoute [CLS] et [SEP]
    max_length=50, # Longueur maximale
    padding='max_length', # Remplit jusqu'à max_length
    truncation=True, # Tronque si plus long que max_length
    return_attention_mask=True,
    return_tensors='pt' # Retourne des tenseurs PyTorch
)

print(f"Input IDs (BERT): {bert_encoded_plus['input_ids']}")
print(f"Attention Mask (BERT): {bert_encoded_plus['attention_mask']}")

# Décodage pour vérifier
decoded_bert = bert_tokenizer.decode(bert_encoded_plus['input_ids'][0], skip_special_tokens=True)
print(f"Décodé (BERT): {decoded_bert}")

# Tokenisation avec XLMRobertaTokenizer
print("\n--- XLMRobertaTokenizer (xlm-roberta-base) ---")
xlm_roberta_tokens = xlm_roberta_tokenizer.tokenize(sentence)
print(f"Tokens (XLM-R): {xlm_roberta_tokens}")

xlm_roberta_encoded_plus = xlm_roberta_tokenizer.encode_plus(
    sentence,
    add_special_tokens=True, # Ajoute <s> et </s>
    max_length=50,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt'
)

print(f"Input IDs (XLM-R): {xlm_roberta_encoded_plus['input_ids']}")
print(f"Attention Mask (XLM-R): {xlm_roberta_encoded_plus['attention_mask']}")

# Décodage pour vérifier
decoded_xlm_r = xlm_roberta_tokenizer.decode(xlm_roberta_encoded_plus['input_ids'][0], skip_special_tokens=True)
print(f"Décodé (XLM-R): {decoded_xlm_r}")

# Exemple de tokenisation de deux phrases (pour BERT)
sentence1 = "Le chat dort sur le tapis."
sentence2 = "Le chien joue dans le jardin."

print("\n--- Tokenisation de deux phrases (BERT) ---")
bert_two_sentences = bert_tokenizer.encode_plus(
    sentence1,
    sentence2,
    add_special_tokens=True,
    max_length=50,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_token_type_ids=True,
    return_tensors='pt'
)

print(f"Input IDs (deux phrases BERT): {bert_two_sentences['input_ids']}")
print(f"Attention Mask (deux phrases BERT): {bert_two_sentences['attention_mask']}")
print(f"Token Type IDs (deux phrases BERT): {bert_two_sentences['token_type_ids']}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]


--- BertTokenizer (bert-base-uncased) ---
Tokens (BERT): ['this', 'is', 'an', 'example', 'sentence', 'for', 'token', '##ization', '.']


AttributeError: BertTokenizer has no attribute encode_plus

## 3. Préparation des données d'entrée pour le modèle

Pour que les modèles Transformers puissent traiter le texte, les entrées doivent être formatées d'une manière spécifique. Cela implique le remplissage (`padding`), la troncation (`truncation`) et l'ajout de jetons spéciaux (`special tokens`).

-   **Jetons spéciaux** : BERT utilise `[CLS]` au début et `[SEP]` pour séparer les phrases ou marquer la fin d'une seule phrase. XLM-RoBERTa utilise `<s>` et `</s>`.
-   **`input_ids`** : La séquence numérique de jetons après tokenisation.
-   **`attention_mask`** : Un masque qui indique au modèle quels jetons doivent être traités (1) et quels sont les jetons de remplissage à ignorer (0).
-   **`token_type_ids`** (pour BERT) : Utilisé lors de la tokenisation de paires de phrases pour indiquer à quelle phrase appartient chaque jeton (0 pour la première phrase, 1 pour la seconde).

In [5]:
from transformers import BertTokenizer

# Re-initialiser le tokenizer BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Exemple de phrases
sentence1 = "Ceci est la première phrase."
sentence2 = "Et ceci est la deuxième phrase, qui est un peu plus longue pour démontrer la troncation et le remplissage."

# Définir la longueur maximale
max_length = 30 # Une petite longueur pour forcer le padding/truncation

print(f"Longueur maximale définie : {max_length}\n")

# Préparation d'une seule phrase
print("--- Préparation d'une seule phrase ---")
encoding_single = tokenizer(
    sentence1,
    add_special_tokens=True,
    max_length=max_length,
    padding='max_length', # Remplit jusqu'à max_length
    truncation=True, # Tronque si plus long que max_length
    return_attention_mask=True,
    return_tensors='pt'
)

print(f"Input IDs (une phrase): {encoding_single['input_ids']}")
print(f"Attention Mask (une phrase): {encoding_single['attention_mask']}")
print(f"Tokens spéciaux (BERT): {tokenizer.special_tokens_map}")
print(f"Taille du vocabulaire (BERT): {tokenizer.vocab_size}")

# Préparation de deux phrases
print("\n--- Préparation de deux phrases ---")
encoding_pair = tokenizer(
    sentence1,
    sentence2,
    add_special_tokens=True,
    max_length=max_length,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_token_type_ids=True, # Important pour les paires de phrases
    return_tensors='pt'
)

print(f"Input IDs (paire de phrases): {encoding_pair['input_ids']}")
print(f"Attention Mask (paire de phrases): {encoding_pair['attention_mask']}")
print(f"Token Type IDs (paire de phrases): {encoding_pair['token_type_ids']}")

# Décodage pour voir le résultat de la troncation/padding
decoded_single = tokenizer.decode(encoding_single['input_ids'][0], skip_special_tokens=False)
print(f"\nDécodé (une phrase avec jetons spéciaux): {decoded_single}")

decoded_pair = tokenizer.decode(encoding_pair['input_ids'][0], skip_special_tokens=False)
print(f"Décodé (paire de phrases avec jetons spéciaux): {decoded_pair}")

Longueur maximale définie : 30

--- Préparation d'une seule phrase ---
Input IDs (une phrase): tensor([[ 101, 8292, 6895, 9765, 2474, 6765, 7655, 1012,  102,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0]])
Attention Mask (une phrase): tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0]])
Tokens spéciaux (BERT): {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
Taille du vocabulaire (BERT): 30522

--- Préparation de deux phrases ---
Input IDs (paire de phrases): tensor([[  101,  8292,  6895,  9765,  2474,  6765,  7655,  1012,   102,  3802,
          8292,  6895,  9765,  2474, 24756,  2666,  4168,  7655,  1010, 21864,
          9765,  4895, 21877,  2226,  4606,  2146,  5657, 10364,  5698,   102]])
Attention Mask (paire de phrases): tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

## 4. Chargement et exploration du jeu de données

Nous allons maintenant charger les fichiers CSV fournis et examiner leur structure pour comprendre les données avec lesquelles nous travaillerons. Le fichier `train.csv` contient les données d'entraînement, et nous allons nous concentrer sur celui-ci pour cette étape.

In [3]:
import pandas as pd

# Charger les jeux de données
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')
sample_submission_df = pd.read_csv('/content/sample_submission.csv')

print("--- Aperçu du jeu de données d'entraînement (train.csv) ---")
display(train_df.head())
print(f"Forme du jeu de données d'entraînement: {train_df.shape}\n")

print("--- Aperçu du jeu de données de test (test.csv) ---")
display(test_df.head())
print(f"Forme du jeu de données de test: {test_df.shape}\n")

print("--- Aperçu du fichier de soumission d'exemple (sample_submission.csv) ---")
display(sample_submission_df.head())
print(f"Forme du fichier de soumission d'exemple: {sample_submission_df.shape}\n")

print("--- Informations sur les colonnes du jeu de données d'entraînement ---")
train_df.info()

--- Aperçu du jeu de données d'entraînement (train.csv) ---


,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
2,3931fbe82a,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,fr,French,0
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
4,86aaa48b45,ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...,เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร,th,Thai,1


Forme du jeu de données d'entraînement: (12120, 6)

--- Aperçu du jeu de données de test (test.csv) ---


,id,premise,hypothesis,lang_abv,language
0,c6d58c3f69,بکس، کیسی، راہیل، یسعیاہ، کیلی، کیلی، اور کولم...,"کیسی کے لئے کوئی یادگار نہیں ہوگا, کولمین ہائی...",ur,Urdu
1,cefcc82292,هذا هو ما تم نصحنا به.,عندما يتم إخبارهم بما يجب عليهم فعله ، فشلت ال...,ar,Arabic
2,e98005252c,et cela est en grande partie dû au fait que le...,Les mères se droguent.,fr,French
3,58518c10ba,与城市及其他公民及社区组织代表就IMA的艺术发展进行对话&amp,IMA与其他组织合作，因为它们都依靠共享资金。,zh,Chinese
4,c32b0d16df,Она все еще была там.,"Мы думали, что она ушла, однако, она осталась.",ru,Russian


Forme du jeu de données de test: (5195, 5)

--- Aperçu du fichier de soumission d'exemple (sample_submission.csv) ---


,id,prediction
0,c6d58c3f69,1
1,cefcc82292,1
2,e98005252c,1
3,58518c10ba,1
4,c32b0d16df,1


Forme du fichier de soumission d'exemple: (5195, 2)

--- Informations sur les colonnes du jeu de données d'entraînement ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12120 entries, 0 to 12119
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          12120 non-null  object
 1   premise     12120 non-null  object
 2   hypothesis  12120 non-null  object
 3   lang_abv    12120 non-null  object
 4   language    12120 non-null  object
 5   label       12120 non-null  int64 
dtypes: int64(1), object(5)
memory usage: 568.3+ KB


D'après `train_df.head()` et `train_df.info()`, les colonnes pertinentes pour l'entraînement du modèle sont probablement `text` (pour l'entrée textuelle) et `label` (pour la catégorie à prédire).

## 5. Création des plis de validation croisée

Nous allons utiliser la **validation croisée stratifiée (Stratified K-Fold)** pour créer des ensembles d'entraînement et de validation. Stratified K-Fold garantit que la distribution des classes (`label`) est maintenue dans chaque pli, ce qui est crucial pour les jeux de données déséquilibrés. Nous allons créer 5 plis.

In [6]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

# Définir le nombre de plis
n_splits = 5

# Initialiser StratifiedKFold
kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Listes pour stocker les indices des plis d'entraînement et de validation
train_folds = []
val_folds = []

print(f"Création de {n_splits} plis de validation croisée stratifiée...\n")

# Itérer sur les plis générés
for fold, (train_index, val_index) in enumerate(kf.split(train_df['premise'], train_df['label'])):
    print(f"--- Pli {fold + 1}/{n_splits} ---")

    # Obtenir les sous-ensembles d'entraînement et de validation pour ce pli
    train_subset = train_df.iloc[train_index]
    val_subset = train_df.iloc[val_index]

    # Stocker les indices
    train_folds.append(train_index)
    val_folds.append(val_index)

    print(f"Taille de l'ensemble d'entraînement: {len(train_index)}")
    print(f"Taille de l'ensemble de validation: {len(val_index)}")

    # Vérifier la distribution des classes dans l'ensemble d'entraînement
    print("Distribution des classes (Entraînement):\n", train_subset['label'].value_counts(normalize=True))
    # Vérifier la distribution des classes dans l'ensemble de validation
    print("Distribution des classes (Validation):\n", val_subset['label'].value_counts(normalize=True))
    print("\n")

print("Processus de création des plis terminé. Les indices sont stockés dans 'train_folds' et 'val_folds'.")
print(f"Nombre total de plis créés: {len(train_folds)}")

Création de 5 plis de validation croisée stratifiée...

--- Pli 1/5 ---
Taille de l'ensemble d'entraînement: 9696
Taille de l'ensemble de validation: 2424
Distribution des classes (Entraînement):
 label
0    0.344472
2    0.335396
1    0.320132
Name: proportion, dtype: float64
Distribution des classes (Validation):
 label
0    0.344884
2    0.334983
1    0.320132
Name: proportion, dtype: float64


--- Pli 2/5 ---
Taille de l'ensemble d'entraînement: 9696
Taille de l'ensemble de validation: 2424
Distribution des classes (Entraînement):
 label
0    0.344575
2    0.335293
1    0.320132
Name: proportion, dtype: float64
Distribution des classes (Validation):
 label
0    0.344472
2    0.335396
1    0.320132
Name: proportion, dtype: float64


--- Pli 3/5 ---
Taille de l'ensemble d'entraînement: 9696
Taille de l'ensemble de validation: 2424
Distribution des classes (Entraînement):
 label
0    0.344575
2    0.335293
1    0.320132
Name: proportion, dtype: float64
Distribution des classes (Valida

## 1. Comprendre BERT et XLM-RoBERTa (Mode Débutant)

Bienvenue dans le monde fascinant des modèles de langage ! Aujourd'hui, nous allons découvrir deux modèles très puissants : **BERT** et **XLM-RoBERTa**.

### Qu'est-ce qu'un modèle de langage ?

Imaginez un programme informatique capable de 'comprendre' le langage humain, de lire des textes, et même de générer de nouvelles phrases. C'est ce qu'on appelle un modèle de langage. Il apprend à reconnaître des motifs et des relations dans les mots pour traiter le texte de manière intelligente.

### Introduction à BERT (Bidirectional Encoder Representations from Transformers)

**BERT** est un modèle créé par Google. C'est comme un étudiant très assidu qui a lu une immense quantité de livres (tout Internet, en fait !) pour apprendre la langue. Voici ses caractéristiques principales :

*   **'Bidirectionnel'** : C'est important ! Cela signifie que BERT lit une phrase dans les deux sens : de gauche à droite ET de droite à gauche. Pourquoi est-ce utile ? Prenons la phrase "La banque de la rivière". Si le modèle ne lit que de gauche à droite, il pourrait penser à une banque où l'on dépose de l'argent. Mais s'il voit le mot 'rivière' après 'banque', il comprendra que 'banque' fait référence au bord de l'eau. BERT utilise ce contexte complet pour mieux comprendre chaque mot.

*   **'Encoder'** : BERT est un 'encodeur'. Son rôle est de prendre des mots en entrée et de les transformer en une représentation numérique que l'ordinateur peut comprendre et manipuler. Pensez-y comme traduire des mots en un code secret très sophistiqué.

*   **'Transformers'** : C'est le type d'architecture réseau neuronal que BERT utilise. Les Transformers sont très bons pour gérer de longues séquences de mots grâce à un mécanisme appelé 'attention', qui permet au modèle de se concentrer sur les mots les plus importants dans une phrase pour une tâche donnée.

*   **Tâches d'apprentissage de BERT** : Pour apprendre le langage, BERT s'entraîne sur deux tâches principales :
    1.  **Masquage de Langue (Masked Language Model - MLM)** : On cache (on 'masque') certains mots dans une phrase, et BERT doit deviner quels étaient ces mots. Par exemple, si on lui donne "Le [MASQUE] dort sur le tapis", il apprendra que 'chat' est un bon candidat.
    2.  **Prédiction de la Prochaine Phrase (Next Sentence Prediction - NSP)** : BERT doit décider si deux phrases sont logiquement liées ou non. Par exemple, est-ce que "Le chat dort." est suivi de "Il fait chaud."? Ou de "Le chien aboie."? Cela l'aide à comprendre la cohérence des textes.

### Introduction à XLM-RoBERTa (Cross-lingual Language Model - Robustly Optimized BERT Pretraining Approach)

**XLM-RoBERTa** est un modèle très similaire à BERT, mais avec un super-pouvoir supplémentaire : il est **multilingue** !

*   **'Cross-lingual' (Translinguistique)** : Cela signifie qu'il peut travailler avec de nombreuses langues différentes en même temps (plus de 100 !). Il a été entraîné sur d'énormes quantités de texte dans toutes ces langues. C'est incroyablement utile car il peut apprendre des choses dans une langue (par exemple, l'anglais) et les appliquer à une autre (comme le français), même si elles sont très différentes.

*   **Utilité du multilingue** : Imaginez que vous voulez analyser des sentiments dans une langue rare pour laquelle vous avez peu de données. XLM-RoBERTa, grâce à son apprentissage sur de nombreuses langues, peut mieux s'en sortir qu'un modèle qui n'aurait appris qu'une seule langue.

### Le rôle de la Tokenisation

Avant que BERT ou XLM-RoBERTa puissent faire quoi que ce soit, le texte doit être 'découpé' en unités plus petites appelées **tokens**. C'est le processus de **tokenisation**. Les tokens peuvent être des mots, des parties de mots, ou même des symboles de ponctuation. C'est la première étape cruciale pour transformer le langage humain en quelque chose que le modèle peut traiter numériquement.

## 2. Tokenisation de texte (Mode Débutant)

Maintenant que nous savons ce qu'est la tokenisation, voyons comment les modèles BERT et XLM-RoBERTa découpent concrètement nos phrases. Nous allons utiliser des outils spéciaux appelés 'tokenizers' de la bibliothèque `transformers`.

### Qu'est-ce qu'un Tokenizer ?

Un **tokenizer** est un programme qui prend votre texte (une phrase, un paragraphe) et le transforme en une liste de 'tokens' (des mots ou des morceaux de mots) et d'autres informations numériques que le modèle attend.

### Les informations clés que le tokenizer nous donne :

Quand un tokenizer traite une phrase, il nous donne généralement plusieurs choses :

1.  **`input_ids`** : C'est la liste la plus importante. Chaque 'token' de votre phrase est remplacé par un numéro unique. Par exemple, le mot 'chat' pourrait devenir le numéro 2053, 'dort' le numéro 4567, etc. C'est la version numérique de votre texte que le modèle va lire.
2.  **`attention_mask`** : C'est une liste de 1 et de 0. Elle indique au modèle quels numéros (`input_ids`) sont de vrais mots de la phrase (marqués par 1) et quels sont juste du 'remplissage' (marqués par 0). Le remplissage est utilisé pour que toutes les phrases aient la même longueur, comme nous le verrons plus tard.
3.  **`token_type_ids`** (spécifique à BERT pour deux phrases) : Quand on donne deux phrases à BERT en même temps, cette liste de 0 et de 1 indique au modèle quels tokens appartiennent à la première phrase (marqués par 0) et quels tokens appartiennent à la deuxième phrase (marqués par 1). C'est crucial pour des tâches comme la prédiction de la prochaine phrase.

### Démonstration avec `BertTokenizer` et `XLMRobertaTokenizer`

Nous allons d'abord importer les tokenizers, puis les utiliser sur quelques phrases.

In [7]:
# Importons les outils nécessaires pour la tokenisation
from transformers import BertTokenizer, XLMRobertaTokenizer

# --- Étape 1 : Charger les Tokenizers ---
# C'est comme choisir l'outil de découpe adapté à notre texte.
# 'bert-base-uncased' est une version de BERT qui ne fait pas la distinction entre majuscules et minuscules.
# 'xlm-roberta-base' est la version de base de XLM-RoBERTa, qui gère plusieurs langues.
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

# --- Étape 2 : Préparer nos phrases d'exemple ---
# Nous allons utiliser une phrase française et son équivalent anglais pour montrer la différence.
sentence_fr = "Ceci est un exemple de phrase pour la tokenisation."
sentence_en = "This is an example sentence for tokenization."

print("--------------------------------------------------")
print("Démonstration avec BertTokenizer (pour l'anglais)")
print("--------------------------------------------------")

# --- Étape 3 : Tokenisation d'une seule phrase avec BertTokenizer ---
# Le BertTokenizer est généralement entraîné sur l'anglais, utilisons donc la phrase anglaise.
# La méthode `.tokenize()` nous donne les tokens 'bruts'.
bert_raw_tokens = bert_tokenizer.tokenize(sentence_en)
print(f"Tokens 'bruts' (BERT): {bert_raw_tokens}")
# Remarquez comment 'tokenization' est divisé en 'token' et '##ization'. C'est typique de BERT.

# La méthode la plus courante pour obtenir toutes les informations numériques est d'appeler le tokenizer directement.
# Elle retourne un 'dictionnaire' avec input_ids, attention_mask, etc.
bert_encoded_output_single = bert_tokenizer(
    sentence_en,                       # Notre phrase à traiter
    add_special_tokens=True,           # Ajoute des tokens spéciaux (comme [CLS] et [SEP])
    max_length=50,                     # Longueur maximale que nos listes de numéros doivent avoir
    padding='max_length',              # Remplit la liste avec des zéros si elle est trop courte
    truncation=True,                   # Coupe la liste si elle est trop longue
    return_attention_mask=True,        # Demande de créer l'attention mask
    return_tensors='pt'                # Demande à retourner des 'tensors' PyTorch (format préféré des modèles)
)

print(f"\nInput IDs (BERT - phrase unique): {bert_encoded_output_single['input_ids']}")
print(f"Attention Mask (BERT - phrase unique): {bert_encoded_output_single['attention_mask']}")

# --- Étape 4 : Décodage pour vérifier (BertTokenizer) ---
# Pour s'assurer que notre tokenisation est correcte, nous pouvons 'reconvertir' les numéros en mots.
# `skip_special_tokens=True` ignore les tokens comme [CLS] et [SEP] pour un texte plus propre.
decoded_bert_single = bert_tokenizer.decode(bert_encoded_output_single['input_ids'][0], skip_special_tokens=True)
print(f"Phrase décodée (BERT - sans tokens spéciaux): {decoded_bert_single}")


print("\n----------------------------------------------------")
print("Démonstration avec XLMRobertaTokenizer (pour le français)")
print("----------------------------------------------------")

# --- Étape 5 : Tokenisation d'une seule phrase avec XLMRobertaTokenizer ---
# XLMRoberta est multilingue, donc nous pouvons utiliser notre phrase française.
xlm_roberta_raw_tokens = xlm_roberta_tokenizer.tokenize(sentence_fr)
print(f"Tokens 'bruts' (XLM-R): {xlm_roberta_raw_tokens}")
# Remarquez les symboles ' ' (barre basse) qui indiquent le début d'un mot ou d'une partie de mot.

xlm_roberta_encoded_output_single = xlm_roberta_tokenizer(
    sentence_fr,                       # Notre phrase française
    add_special_tokens=True,           # Ajoute des tokens spéciaux (comme <s> et </s>)
    max_length=50,                     # Même longueur maximale
    padding='max_length',              # Remplit avec des zéros
    truncation=True,                   # Coupe si trop long
    return_attention_mask=True,
    return_tensors='pt'
)

print(f"\nInput IDs (XLM-R - phrase unique): {xlm_roberta_encoded_output_single['input_ids']}")
print(f"Attention Mask (XLM-R - phrase unique): {xlm_roberta_encoded_output_single['attention_mask']}")

# --- Étape 6 : Décodage pour vérifier (XLMRobertaTokenizer) ---
decoded_xlm_r_single = xlm_roberta_tokenizer.decode(xlm_roberta_encoded_output_single['input_ids'][0], skip_special_tokens=True)
print(f"Phrase décodée (XLM-R - sans tokens spéciaux): {decoded_xlm_r_single}")


print("\n------------------------------------------------")
print("Tokenisation de deux phrases (avec BertTokenizer)")
print("------------------------------------------------")

# --- Étape 7 : Tokenisation de deux phrases avec BertTokenizer ---
# BERT peut traiter deux phrases en même temps, ce qui est utile pour des tâches de comparaison.
sentence1_pair = "Le chat dort sur le tapis."
sentence2_pair = "Le chien joue dans le jardin."

bert_encoded_output_pair = bert_tokenizer(
    sentence1_pair,
    sentence2_pair,                    # La deuxième phrase
    add_special_tokens=True,
    max_length=50,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_token_type_ids=True,        # Très important pour les paires de phrases de BERT
    return_tensors='pt'
)

print(f"\nInput IDs (BERT - deux phrases): {bert_encoded_output_pair['input_ids']}")
print(f"Attention Mask (BERT - deux phrases): {bert_encoded_output_pair['attention_mask']}")
print(f"Token Type IDs (BERT - deux phrases): {bert_encoded_output_pair['token_type_ids']}")
# Le `token_type_ids` montre 0 pour la première phrase et 1 pour la deuxième, et 0 pour les tokens spéciaux entre les deux.

# Décodons pour voir comment les deux phrases sont représentées avec les tokens spéciaux.
decoded_bert_pair = bert_tokenizer.decode(bert_encoded_output_pair['input_ids'][0], skip_special_tokens=False)
print(f"Phrase décodée (BERT - deux phrases avec tokens spéciaux): {decoded_bert_pair}")

--------------------------------------------------
Démonstration avec BertTokenizer (pour l'anglais)
--------------------------------------------------
Tokens 'bruts' (BERT): ['this', 'is', 'an', 'example', 'sentence', 'for', 'token', '##ization', '.']

Input IDs (BERT - phrase unique): tensor([[  101,  2023,  2003,  2019,  2742,  6251,  2005, 19204,  3989,  1012,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0]])
Attention Mask (BERT - phrase unique): tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0]])
Phrase décodée (BERT - sans tokens spéciaux): this is an example sentence for tokenization.

----

## 3. Préparation des données d'entrée pour le modèle (Mode Débutant)

Maintenant que nous savons tokeniser le texte, nous allons nous concentrer sur la manière de préparer ces tokens pour qu'un modèle Transformer puisse les utiliser. Il y a trois concepts clés à maîtriser : le **remplissage (padding)**, la **troncation (truncation)** et les **tokens spéciaux**.

### 1. Remplissage (Padding)

*   **Problème** : Les ordinateurs sont plus efficaces quand ils traitent des données de taille fixe. Or, nos phrases ont des longueurs différentes (une phrase courte et une très longue).
*   **Solution** : Le remplissage consiste à ajouter des 'tokens de remplissage' (souvent le token `[PAD]`, représenté par le numéro 0) à la fin des phrases trop courtes pour qu'elles atteignent une `max_length` (longueur maximale) prédéfinie. Toutes les séquences d'entrée auront ainsi la même taille.
*   **Rôle de l'`attention_mask`** : L'`attention_mask` est une liste de 1 et 0. Pour les vrais tokens de la phrase, il y a un 1. Pour les tokens de remplissage (les `[PAD]`), il y a un 0. Cela indique au modèle d'ignorer ces tokens de remplissage pour ne pas les prendre en compte dans ses calculs.

### 2. Troncation (Truncation)

*   **Problème** : Si une phrase est trop longue et dépasse la `max_length` que nous avons fixée, le modèle ne peut pas traiter tous les tokens.
*   **Solution** : La troncation consiste à couper la phrase à partir de la `max_length`. Le reste de la phrase est ignoré. C'est un compromis pour gérer des textes très longs, mais cela peut faire perdre des informations.

### 3. Tokens Spéciaux

Les modèles Transformers utilisent des tokens spéciaux pour marquer le début, la fin, ou la séparation de phrases. Ils sont essentiels pour le bon fonctionnement du modèle :

*   **Pour BERT** :
    *   `[CLS]` : (Classifier) Toujours placé au **début** de la première phrase. Son rôle est de collecter toutes les informations sur la phrase ou la paire de phrases et de servir de représentation globale pour des tâches de classification.
    *   `[SEP]` : (Separator) Placé à la **fin** de chaque phrase. Si vous avez deux phrases, un `[SEP]` est inséré entre elles, puis un autre à la toute fin.
    *   `[PAD]` : (Padding) Comme expliqué ci-dessus, utilisé pour le remplissage.
    *   `[UNK]` : (Unknown) Utilisé pour remplacer les mots que le modèle n'a jamais vus pendant son entraînement.
    *   `[MASK]` : Utilisé pendant l'entraînement pour la tâche de masquage de langue (MLM).

*   **Pour XLM-RoBERTa** :
    *   `<s>` : Souvent utilisé comme marqueur de **début** de séquence.
    *   `</s>` : Souvent utilisé comme marqueur de **fin** de séquence.
    *   Les autres tokens comme `[PAD]` et `[UNK]` existent aussi.

### Démonstration de la préparation des données

Nous allons utiliser un `BertTokenizer` pour illustrer ces concepts avec des phrases de différentes longueurs et une `max_length` courte pour bien voir les effets du remplissage et de la troncation.

In [8]:
from transformers import BertTokenizer

# --- Étape 1 : Charger le Tokenizer BERT ---
# Nous réutilisons le tokenizer BERT de l'exercice précédent.
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# --- Étape 2 : Définir des phrases d'exemple ---
# Une phrase courte et une plus longue pour bien voir le padding et la truncation.
sentence_short = "Ceci est une courte phrase."
sentence_long = "Ceci est une très très très très très très très très très très très très très très très longue phrase pour démontrer la troncation et le remplissage."

# --- Étape 3 : Définir la longueur maximale (max_length) ---
# C'est la taille que toutes nos séquences d'entrée devront avoir.
# Nous la mettons volontairement petite pour observer les effets.
max_length = 20
print(f"Longueur maximale définie (max_length) : {max_length} tokens\n")

print("----------------------------------------")
print("Préparation d'une SEULE phrase courte")
print("----------------------------------------")

# --- Étape 4 : Préparation d'une phrase courte ---
# Nous utilisons le tokenizer directement. Les options 'padding' et 'truncation' sont essentielles ici.
encoding_short = tokenizer(
    sentence_short,             # La phrase à encoder
    add_special_tokens=True,    # Ajoute [CLS] et [SEP]
    max_length=max_length,      # Toutes les séquences doivent faire 20 tokens
    padding='max_length',       # Ajoute [PAD] (0) jusqu'à max_length
    truncation=True,            # Coupe la phrase si elle dépasse max_length
    return_attention_mask=True, # Génère l'attention mask
    return_tensors='pt'         # Retourne des tenseurs PyTorch
)

print(f"Input IDs (phrase courte): {encoding_short['input_ids']}")
print(f"Attention Mask (phrase courte): {encoding_short['attention_mask']}")

# Observons les tokens spéciaux et la taille du vocabulaire du tokenizer
print(f"Tokens spéciaux utilisés par BERT : {tokenizer.special_tokens_map}")
print(f"Taille du vocabulaire BERT (nombre de mots/fragments connus) : {tokenizer.vocab_size}")

# Décodons pour voir comment la phrase courte est remplie
decoded_short_padded = tokenizer.decode(encoding_short['input_ids'][0], skip_special_tokens=False)
print(f"\nPhrase décodée (avec padding) : {decoded_short_padded}")
print("--> Remarquez les '[PAD]' à la fin : c'est le remplissage !")

print("\n----------------------------------------")
print("Préparation d'une SEULE phrase longue")
print("----------------------------------------")

# --- Étape 5 : Préparation d'une phrase longue ---
# Même processus, mais cette fois la phrase est trop longue.
encoding_long = tokenizer(
    sentence_long,              # La phrase longue à encoder
    add_special_tokens=True,
    max_length=max_length,      # La phrase sera coupée à cette longueur
    padding='max_length',
    truncation=True,            # Active la troncation
    return_attention_mask=True,
    return_tensors='pt'
)

print(f"Input IDs (phrase longue): {encoding_long['input_ids']}")
print(f"Attention Mask (phrase longue): {encoding_long['attention_mask']}")

# Décodons pour voir comment la phrase longue est tronquée
decoded_long_truncated = tokenizer.decode(encoding_long['input_ids'][0], skip_special_tokens=False)
print(f"\nPhrase décodée (avec troncation) : {decoded_long_truncated}")
print("--> Remarquez que la fin de la phrase a été coupée et un '[SEP]' a été ajouté.")

print("\n----------------------------------------")
print("Préparation d'une PAIRE de phrases")
print("----------------------------------------")

# --- Étape 6 : Préparation d'une paire de phrases ---
# Nous utilisons deux phrases. BERT utilise des 'token_type_ids' pour distinguer la première et la seconde.
sentence1_pair = "Le chat dort sur le tapis."
sentence2_pair = "Le chien joue dans le jardin."

encoding_pair = tokenizer(
    sentence1_pair,
    sentence2_pair,             # La deuxième phrase est passée comme un deuxième argument
    add_special_tokens=True,
    max_length=max_length,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_token_type_ids=True, # TRÈS IMPORTANT pour les paires de phrases avec BERT
    return_tensors='pt'
)

print(f"Input IDs (paire de phrases): {encoding_pair['input_ids']}")
print(f"Attention Mask (paire de phrases): {encoding_pair['attention_mask']}")
print(f"Token Type IDs (paire de phrases): {encoding_pair['token_type_ids']}")

# Décodons la paire de phrases
decoded_pair_combined = tokenizer.decode(encoding_pair['input_ids'][0], skip_special_tokens=False)
print(f"\nPhrase décodée (paire de phrases) : {decoded_pair_combined}")
print("--> Le '[SEP]' sépare les deux phrases, et les 'token_type_ids' indiquent quelle phrase chaque token représente.")


Longueur maximale définie (max_length) : 20 tokens

----------------------------------------
Préparation d'une SEULE phrase courte
----------------------------------------
Input IDs (phrase courte): tensor([[  101,  8292,  6895,  9765, 16655,  2457,  2063,  7655,  1012,   102,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0]])
Attention Mask (phrase courte): tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
Tokens spéciaux utilisés par BERT : {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
Taille du vocabulaire BERT (nombre de mots/fragments connus) : 30522

Phrase décodée (avec padding) : [CLS] ceci est une courte phrase. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
--> Remarquez les '[PAD]' à la fin : c'est le remplissage !

----------------------------------------
Préparation d'une SEULE phrase longue
----------------------------------------
Input 

## 4. Chargement et exploration du jeu de données (Mode Débutant)

Maintenant que nous savons comment préparer le texte, la prochaine étape est de charger les données réelles avec lesquelles nous allons travailler. Ces données sont stockées dans des fichiers CSV (Comma Separated Values), qui sont un format courant pour les tableaux de données.

### Qu'est-ce qu'un fichier CSV ?

Imaginez une feuille de calcul Excel très simple, où les colonnes sont séparées par des virgules. C'est ça, un fichier CSV. Chaque ligne représente une entrée de données.

### L'outil `pandas`

Nous allons utiliser la bibliothèque `pandas` en Python. C'est un outil très puissant pour travailler avec des tableaux de données (appelés 'DataFrames').

### Les fonctions clés à utiliser :

1.  **`pd.read_csv('/chemin/vers/votre/fichier.csv')`** : Cette fonction lit un fichier CSV et le transforme en un DataFrame `pandas`.
2.  **`df.head()`** : Une fois que vous avez un DataFrame (que nous appelons souvent `df`), cette fonction affiche les 5 premières lignes. C'est très utile pour avoir un aperçu rapide des données et comprendre leur structure.
3.  **`df.shape`** : Cette propriété vous donne la 'forme' de votre DataFrame, c'est-à-dire le nombre de lignes et le nombre de colonnes. Par exemple, `(100, 5)` signifierait 100 lignes et 5 colonnes.
4.  **`df.info()`** : Cette fonction fournit un résumé concis du DataFrame, y compris le nombre d'entrées non nulles par colonne, le type de données de chaque colonne (par exemple, texte, nombre entier), et l'utilisation de la mémoire. C'est très utile pour vérifier la qualité des données et identifier les problèmes potentiels (comme des valeurs manquantes).

### Chargement et exploration

Nous allons charger les fichiers `train.csv`, `test.csv` et `sample_submission.csv` et les explorer.

In [9]:
# Importons la bibliothèque pandas, que nous allons appeler 'pd' pour faire court.
import pandas as pd

# --- Étape 1 : Charger les jeux de données CSV ---
# Nous utilisons pd.read_csv pour lire nos fichiers de données.
# Les chemins des fichiers sont fournis comme '/content/nom_du_fichier.csv'.
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')
sample_submission_df = pd.read_csv('/content/sample_submission.csv')

print("----------------------------------------------------------")
print("Aperçu et informations du jeu de données d'entraînement (train.csv)")
print("----------------------------------------------------------")

# --- Étape 2 : Afficher les premières lignes du DataFrame d'entraînement ---
# C'est comme regarder les 5 premières lignes d'une feuille de calcul.
print("Premières 5 lignes du DataFrame d'entraînement :")
display(train_df.head())

# --- Étape 3 : Afficher la forme (dimensions) du DataFrame d'entraînement ---
# Cela nous indique combien de lignes et de colonnes il y a.
print(f"\nForme du jeu de données d'entraînement (lignes, colonnes) : {train_df.shape}")

# --- Étape 4 : Afficher un résumé des informations du DataFrame d'entraînement ---
# Cela nous donne des détails sur chaque colonne : son nom, le nombre de valeurs non nulles, et son type de données.
print("\nInformations détaillées sur le DataFrame d'entraînement :")
train_df.info()


print("\n----------------------------------------------------")
print("Aperçu et informations du jeu de données de test (test.csv)")
print("----------------------------------------------------")

# --- Étape 5 : Afficher les premières lignes et la forme du DataFrame de test ---
print("Premières 5 lignes du DataFrame de test :")
display(test_df.head())
print(f"\nForme du jeu de données de test (lignes, colonnes) : {test_df.shape}")


print("\n------------------------------------------------------------")
print("Aperçu et informations du fichier de soumission d'exemple (sample_submission.csv)")
print("------------------------------------------------------------")

# --- Étape 6 : Afficher les premières lignes et la forme du DataFrame de soumission d'exemple ---
print("Premières 5 lignes du DataFrame de soumission d'exemple :")
display(sample_submission_df.head())
print(f"\nForme du fichier de soumission d'exemple (lignes, colonnes) : {sample_submission_df.shape}")

print("\n--- Identification des colonnes clés ---")
print("D'après les aperçus, pour le jeu de données d'entraînement (`train_df`):")
print("- La colonne 'premise' contient le texte d'entrée (nos phrases à analyser).")
print("- La colonne 'label' contient la catégorie ou la classe que nous voulons prédire (0, 1, 2). C'est notre variable cible.")
print("Pour le jeu de données de test (`test_df`), la colonne 'premise' est également le texte d'entrée. Il n'y a pas de colonne 'label', car c'est ce que notre modèle devra prédire.")

----------------------------------------------------------
Aperçu et informations du jeu de données d'entraînement (train.csv)
----------------------------------------------------------
Premières 5 lignes du DataFrame d'entraînement :


,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
2,3931fbe82a,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,fr,French,0
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
4,86aaa48b45,ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...,เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร,th,Thai,1



Forme du jeu de données d'entraînement (lignes, colonnes) : (12120, 6)

Informations détaillées sur le DataFrame d'entraînement :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12120 entries, 0 to 12119
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          12120 non-null  object
 1   premise     12120 non-null  object
 2   hypothesis  12120 non-null  object
 3   lang_abv    12120 non-null  object
 4   language    12120 non-null  object
 5   label       12120 non-null  int64 
dtypes: int64(1), object(5)
memory usage: 568.3+ KB

----------------------------------------------------
Aperçu et informations du jeu de données de test (test.csv)
----------------------------------------------------
Premières 5 lignes du DataFrame de test :


,id,premise,hypothesis,lang_abv,language
0,c6d58c3f69,بکس، کیسی، راہیل، یسعیاہ، کیلی، کیلی، اور کولم...,"کیسی کے لئے کوئی یادگار نہیں ہوگا, کولمین ہائی...",ur,Urdu
1,cefcc82292,هذا هو ما تم نصحنا به.,عندما يتم إخبارهم بما يجب عليهم فعله ، فشلت ال...,ar,Arabic
2,e98005252c,et cela est en grande partie dû au fait que le...,Les mères se droguent.,fr,French
3,58518c10ba,与城市及其他公民及社区组织代表就IMA的艺术发展进行对话&amp,IMA与其他组织合作，因为它们都依靠共享资金。,zh,Chinese
4,c32b0d16df,Она все еще была там.,"Мы думали, что она ушла, однако, она осталась.",ru,Russian



Forme du jeu de données de test (lignes, colonnes) : (5195, 5)

------------------------------------------------------------
Aperçu et informations du fichier de soumission d'exemple (sample_submission.csv)
------------------------------------------------------------
Premières 5 lignes du DataFrame de soumission d'exemple :


,id,prediction
0,c6d58c3f69,1
1,cefcc82292,1
2,e98005252c,1
3,58518c10ba,1
4,c32b0d16df,1



Forme du fichier de soumission d'exemple (lignes, colonnes) : (5195, 2)

--- Identification des colonnes clés ---
D'après les aperçus, pour le jeu de données d'entraînement (`train_df`):
- La colonne 'premise' contient le texte d'entrée (nos phrases à analyser).
- La colonne 'label' contient la catégorie ou la classe que nous voulons prédire (0, 1, 2). C'est notre variable cible.
Pour le jeu de données de test (`test_df`), la colonne 'premise' est également le texte d'entrée. Il n'y a pas de colonne 'label', car c'est ce que notre modèle devra prédire.


## 5. Création des plis de validation croisée (Mode Débutant)

Lorsque nous entraînons un modèle d'apprentissage automatique, nous devons nous assurer qu'il fonctionne bien non seulement sur les données qu'il a 'vues' pendant l'entraînement, mais aussi sur de nouvelles données 'inconnues'. La **validation croisée (Cross-Validation)** est une technique très importante pour cela.

### Pourquoi la Validation Croisée ?

Imaginez que vous êtes un professeur et que vous voulez évaluer un élève. Si vous lui donnez toujours les mêmes questions d'examen qu'il a déjà étudiées, vous ne saurez pas s'il a vraiment compris la matière ou s'il a juste mémorisé les réponses. La validation croisée, c'est comme donner à l'élève plusieurs examens différents, avec des questions différentes à chaque fois, pour avoir une évaluation plus juste de ses compétences.

### Le concept du 'K-Fold' (K-Plis)

*   **K-Fold Cross-Validation** : L'idée est de diviser l'ensemble de nos données d'entraînement en `K` parties (ou 'plis').
*   **Comment ça marche ?** :
    1.  On choisit un pli pour être l'ensemble de **validation** (les questions d'examen).
    2.  Les `K-1` autres plis sont utilisés comme ensemble d'**entraînement** (les leçons à étudier).
    3.  On entraîne le modèle avec les données d'entraînement et on le teste sur les données de validation.
    4.  On répète ces étapes `K` fois, en choisissant un pli de validation différent à chaque fois. Ainsi, chaque pli aura servi de validation une fois.
*   **Pourquoi faire ça ?** : Cela nous donne une estimation plus robuste de la performance de notre modèle, car il est testé sur différentes parties des données, réduisant le risque qu'il ne fonctionne bien que sur un seul découpage spécifique.

### Stratified K-Fold (Validation Croisée Stratifiée)

*   **Problème** : Si nous avons des catégories (nos 'labels', comme 0, 1, 2) qui sont très déséquilibrées (par exemple, 90% de label 0, 5% de label 1, 5% de label 2), un découpage K-Fold simple pourrait, par hasard, mettre très peu d'exemples de label 1 ou 2 dans un pli de validation. Cela rendrait l'évaluation injuste.
*   **Solution** : **Stratified K-Fold** s'assure que chaque pli (d'entraînement et de validation) a la **même proportion de chaque catégorie (label)** que l'ensemble des données d'origine. C'est comme s'assurer que chaque examen contient une proportion équilibrée de questions sur chaque chapitre du livre.

### Mise en œuvre avec `sklearn`

Nous allons utiliser la fonction `StratifiedKFold` de la bibliothèque `sklearn.model_selection`:

*   **`n_splits`** : Le nombre de plis que nous voulons créer (ici, 5).
*   **`shuffle=True`** : Mélange les données avant de créer les plis. C'est important pour éviter tout ordre préétabli qui pourrait biaiser les résultats.
*   **`random_state`** : Un nombre fixe (comme 42). Cela garantit que le mélange des données est toujours le même chaque fois que vous exécutez le code. C'est utile pour la reproductibilité de vos expériences.

In [10]:
# Importons l'outil StratifiedKFold de scikit-learn
from sklearn.model_selection import StratifiedKFold
import numpy as np # Importé juste au cas où, mais pas directement utilisé pour KFold lui-même ici.

# --- Étape 1 : Définir le nombre de plis (K) ---
n_splits = 5 # Nous allons diviser nos données en 5 parties.

# --- Étape 2 : Initialiser StratifiedKFold ---
# Nous configurons notre "professeur d'examen" pour qu'il mélange les données et s'assure d'une bonne proportion de chaque type de question.
kf = StratifiedKFold(
    n_splits=n_splits,     # Nous voulons 5 plis
    shuffle=True,          # Mélange les données avant de les diviser en plis
    random_state=42        # Assure que le mélange est toujours le même, pour des résultats reproductibles
)

# --- Étape 3 : Préparer des listes pour stocker les indices des plis ---
# Nous allons stocker les numéros de ligne (indices) qui appartiennent à chaque ensemble d'entraînement et de validation.
train_folds_indices = [] # Pour les indices des données d'entraînement de chaque pli
val_folds_indices = []   # Pour les indices des données de validation de chaque pli

print(f"Démarrage de la création de {n_splits} plis de validation croisée stratifiée...\n")

# --- Étape 4 : Itérer sur chaque pli ---
# La méthode kf.split() va générer les indices pour chaque pli.
# Nous lui donnons le texte (train_df['premise']) et les labels (train_df['label']) pour qu'il puisse stratifier correctement.
for fold_num, (train_index, val_index) in enumerate(kf.split(train_df['premise'], train_df['label'])):
    print(f"--- Analyse du Pli Numéro {fold_num + 1}/{n_splits} ---")

    # --- Étape 5 : Obtenir les sous-ensembles de données pour ce pli ---
    # Nous utilisons les indices pour sélectionner les lignes correspondantes dans notre DataFrame original.
    train_subset = train_df.iloc[train_index] # Les données pour entraîner le modèle
    val_subset = train_df.iloc[val_index]     # Les données pour valider le modèle

    # --- Étape 6 : Stocker les indices de ce pli ---
    train_folds_indices.append(train_index)
    val_folds_indices.append(val_index)

    # --- Étape 7 : Afficher les tailles et distributions pour vérification ---
    print(f"Taille de l'ensemble d'entraînement pour ce pli : {len(train_index)} lignes")
    print(f"Taille de l'ensemble de validation pour ce pli : {len(val_index)} lignes")

    # Vérifions que la distribution des 'labels' est similaire dans l'entraînement et la validation
    print("Distribution des classes (labels) dans l'entraînement (en pourcentages) :\n", train_subset['label'].value_counts(normalize=True).sort_index())
    print("Distribution des classes (labels) dans la validation (en pourcentages) :\n", val_subset['label'].value_counts(normalize=True).sort_index())
    print("\n") # Ajoutons une ligne vide pour une meilleure lisibilité entre les plis

print("-------------------------------------------------------------------------")
print("Processus de création des plis terminé !")
print("Les indices de chaque pli sont stockés dans 'train_folds_indices' et 'val_folds_indices'.")
print(f"Nombre total de plis créés : {len(train_folds_indices)}")
print("Vous avez maintenant 5 paires d'ensembles d'entraînement/validation prêts pour l'entraînement de votre modèle !")

Démarrage de la création de 5 plis de validation croisée stratifiée...

--- Analyse du Pli Numéro 1/5 ---
Taille de l'ensemble d'entraînement pour ce pli : 9696 lignes
Taille de l'ensemble de validation pour ce pli : 2424 lignes
Distribution des classes (labels) dans l'entraînement (en pourcentages) :
 label
0    0.344472
1    0.320132
2    0.335396
Name: proportion, dtype: float64
Distribution des classes (labels) dans la validation (en pourcentages) :
 label
0    0.344884
1    0.320132
2    0.334983
Name: proportion, dtype: float64


--- Analyse du Pli Numéro 2/5 ---
Taille de l'ensemble d'entraînement pour ce pli : 9696 lignes
Taille de l'ensemble de validation pour ce pli : 2424 lignes
Distribution des classes (labels) dans l'entraînement (en pourcentages) :
 label
0    0.344575
1    0.320132
2    0.335293
Name: proportion, dtype: float64
Distribution des classes (labels) dans la validation (en pourcentages) :
 label
0    0.344472
1    0.320132
2    0.335396
Name: proportion, dtype